# **Parte 1 – Visualización con Plotly**

### Responsable: Lucia y Jose


#### **Paso 1** Cargar el dataset

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("datos/dataset.csv", dtype={"ubigeo": str})
print("Dimensiones del dataset:", df.shape)
display(df.head())

Dimensiones del dataset: (25, 9)


,ubigeo,departamento,declaratorias,prorrogas,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,01,Amazonas,1,0,Chachapoyas,-6.23169,-77.86903,436.5,1
1,02,Áncash,0,0,Huaraz,-9.52614,-77.52869,975.3,7
2,03,Apurímac,0,0,Abancay,-13.63426,-72.88380,711.8,2
3,04,Arequipa,0,0,Arequipa,-16.39899,-71.53747,189.9,0
4,05,Ayacucho,1,0,Ayacucho,-13.16380,-74.22345,485.1,0


##### **Paso 2.** Pregunta y columnas

Se analizará cómo se distribuyen la lluvia y los decretos de emergencia entre departamentos, y si la lluvia acumulada en sus capitales presenta un patrón según la latitud.

Se utilizarán `departamento`, `declaratorias`, `prorrogas`, `lluvia_total_mm` y `latitud`. La lluvia corresponde a la capital utilizada en el assignment 2, no al promedio departamental.

#### **Paso 3.** Crear gráfico interactivo de barras

In [2]:
decretos = df.assign(total_decretos=df["declaratorias"] + df["prorrogas"])
decretos = decretos.sort_values("total_decretos", ascending=False)

fig_barras = px.bar(
    decretos, x="departamento", y="total_decretos",
    title="Amazonas, Ayacucho y Piura registran un decreto cada uno",
    labels={"departamento": "Departamento", "total_decretos": "Declaratorias y prórrogas"},
    template="plotly_white"
)
fig_barras.update_layout(yaxis_dtick=1, margin=dict(b=140), annotations=[dict(
    text="Fuente: decretos de la PCM en gob.pe · Assignment 2.",
    xref="paper", yref="paper", x=1, y=-0.45, xanchor="right", showarrow=False,
    font=dict(size=10, color="gray")
)])
fig_barras.show()

#### **Paso 4.** Crear gráfico interactivo de dispersión

In [3]:
fig_dispersion = px.scatter(
    df, x="latitud", y="lluvia_total_mm", hover_name="departamento",
    hover_data=["capital"],
    title="La lluvia varía entre capitales incluso en latitudes cercanas",
    labels={"latitud": "Latitud de la capital (grados)",
            "lluvia_total_mm": "Lluvia acumulada en la capital (mm)"},
    template="plotly_white"
)
fig_dispersion.update_layout(margin=dict(b=90), annotations=[dict(
    text="Fuente: Open-Meteo · Assignment 2.",
    xref="paper", yref="paper", x=1, y=-0.20, xanchor="right", showarrow=False,
    font=dict(size=10, color="gray")
)])
fig_dispersion.show()

#### **Paso 5.** Tercer gráfico libre

In [4]:
fig_libre = px.histogram(
    df,
    x="lluvia_total_mm",
    title="Distribución de lluvia acumulada",
    labels={"lluvia_total_mm": "Lluvia acumulada (mm)"},
    template="plotly_white",
    color_discrete_sequence=["#2878A5"]
)

fig_libre.update_traces(
    xbins=dict(start=0, end=1800, size=200),
    texttemplate="%{y}",
    textposition="inside",
    textfont=dict(size=16, color="white")
)

fig_libre.update_layout(
    title=dict(x=0.5, xanchor="center"),
    xaxis=dict(tick0=0, dtick=200, showgrid=False),
    yaxis=dict(visible=False, showgrid=False),
    bargap=0.08,
    height=480
)

fig_libre.show()

Se eligió crear un histograma porque permite identificar de manera sencilla cómo se distribuye la lluvia acumulada entre las capitales departamentales. Al observar el gráfico, se aprecia que la mayoría registra menos de 1000 mm, mientras que pocas presentan valores superiores.

#### **Paso 6.** Comprobar datos

In [5]:
extremos = df.loc[
    [df["lluvia_total_mm"].idxmin(), df["lluvia_total_mm"].idxmax()],
    ["departamento", "capital", "lluvia_total_mm"]
]

display(extremos)

conteo = pd.cut(
    df["lluvia_total_mm"],
    bins=range(0, 1801, 200),
    right=False
).value_counts(sort=False)

display(conteo.rename("Número de departamentos").to_frame())

assert conteo.sum() == len(df), "Hay datos fuera de los intervalos."

,departamento,capital,lluvia_total_mm
22,Tacna,Tacna,11.5
15,Loreto,Iquitos,1581.8


,Número de departamentos
lluvia_total_mm,
"[0, 200)",9
"[200, 400)",1
"[400, 600)",3
"[600, 800)",6
"[800, 1000)",4
"[1000, 1200)",0
"[1200, 1400)",1
"[1400, 1600)",1
"[1600, 1800)",0


In [6]:
display(decretos[["departamento", "total_decretos"]])
assert list(fig_barras.data[0].y) == decretos["total_decretos"].tolist()
assert list(fig_dispersion.data[0].x) == df["latitud"].tolist()
assert list(fig_dispersion.data[0].y) == df["lluvia_total_mm"].tolist()
print("Barras y dispersión: los valores representados coinciden con la tabla.")

,departamento,total_decretos
0,Amazonas,1
4,Ayacucho,1
19,Piura,1
2,Apurímac,0
1,Áncash,0
5,Cajamarca,0
6,Callao,0
7,Cusco,0
3,Arequipa,0
8,Huancavelica,0


Barras y dispersión: los valores representados coinciden con la tabla.


Al revisar los valores extremos de la tabla, se identifica que el departamento con la capital con la menor lluvia acumulada (11,5 mm) y la mayor (1581,8 mm) coinciden con el primer y último intervalo del histograma: 0–200 mm y 1400–1600 mm, respectivamente. Además, el conteo por intervalos permite contrastar las cantidades representadas en las barras y confirmar que los 25 departamentos están incluidos.

#### **Paso 7.** Ajuste de gráficos

In [7]:
fig_libre.update_layout(
    title=dict(
        text=(
            "Distribución de la lluvia en las capitales departamentales"
            "<br><sup>Número de departamentos por rango</sup>"
        ),
        font=dict(size=18)
    ),
    xaxis_title="Lluvia acumulada (mm)",
    margin=dict(t=90, b=90),
    annotations=[dict(
        text="Fuente: Open-Meteo · Assignment 2.",
        xref="paper", yref="paper",
        x=1, y=-0.20,
        xanchor="right",
        showarrow=False,
        font=dict(size=10, color="gray")
    )]
)

fig_libre.show()

Se ajustaron el título y las etiquetas para explicar con mayor claridad qué muestra el gráfico. Así, se distingue la lluvia acumulada, expresada en milímetros, de los números dentro de las barras, que indican cuántos departamentos corresponden a cada rango.

#### **Paso 8.** Exportar a HTML

In [8]:
fig_libre.write_html("grafico.html")

Se exportó el tercer gráfico a HTML por contar con la presentación final ajustada.